# 03 — Model optimisation & training (Step 3)
Train and tune **two models** — a **Neural Network (Keras)** and a **Random Forest (scikit-learn)** — for
each `X` scenario. Hyperparameters are tuned with **cross-validation** (kept light to respect the ~30-min
Colab budget). We record MSE/MAE/R² and **training duration**.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 60            # downsampling step in seconds. Lower = more samples, more time (try 30 or 15).
N_USERS          = 1500          # ACC Arena users to load. Closest-users neighbours are searched WITHIN this subset.
N_USERS_SALT     = 1000          # Salt&Tar users (transfer-learning notebook)
X_VALUES         = [1, 3, 5, 10] # number of closest users to experiment with
BEST_X           = 5             # X used for the transfer-learning experiment (pick the best from notebook 04)
LOG_TARGET       = True          # train on log1p(throughput) (target is very skewed); metrics stay in Mbps
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    return {"MSE": float(mean_squared_error(y_true, y_pred)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2":  float(r2_score(y_true, y_pred))}

In [ ]:
import json, time, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
tf.random.set_seed(RANDOM_SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

def load_xy(x):
    d = np.load(f"{PROCESSED_DIR}/acc_X{x}.npz")
    return d["X_train"], d["y_train"], d["X_val"], d["y_val"], d["X_test"], d["y_test"]

## Neural network
Small MLP. We do a tiny grid over width/learning-rate using the validation set, then keep the best.

In [ ]:
def build_mlp(input_dim, units=64, lr=1e-3, depth=2):
    m = keras.Sequential([keras.layers.Input((input_dim,))])
    for _ in range(depth):
        m.add(keras.layers.Dense(units, activation="relu"))
    m.add(keras.layers.Dense(1))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=["mae"])
    return m

def train_nn(Xtr, ytr, Xva, yva):
    best, best_val, best_cfg = None, np.inf, None
    es = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    for units in (64, 128):
        for lr in (1e-3, 3e-4):
            m = build_mlp(Xtr.shape[1], units=units, lr=lr)
            m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=40, batch_size=512,
                  callbacks=[es], verbose=0)
            v = m.evaluate(Xva, yva, verbose=0)[0]
            if v < best_val:
                best, best_val, best_cfg = m, v, {"units": units, "lr": lr}
    return best, best_cfg

## Random forest
`GridSearchCV` (3-fold) over a small grid satisfies the cross-validation requirement.

In [ ]:
def train_rf(Xtr, ytr):
    grid = {"n_estimators": [100, 200], "max_depth": [None, 20]}
    gs = GridSearchCV(RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1),
                      grid, cv=3, scoring="neg_mean_squared_error", n_jobs=-1)
    gs.fit(Xtr, ytr)
    return gs.best_estimator_, gs.best_params_

## Train both models for every X and save results

In [ ]:
def tf_(y):  return np.log1p(y) if LOG_TARGET else y
def inv_(y): return np.expm1(y) if LOG_TARGET else y

results = []
for x in [0] + X_VALUES:
    Xtr, ytr, Xva, yva, Xte, yte = load_xy(x)

    t = time.time(); nn, nn_cfg = train_nn(Xtr, tf_(ytr), Xva, tf_(yva)); nn_dur = time.time()-t
    nn_pred = inv_(nn.predict(Xte, verbose=0).ravel())
    nn_m = evaluate(yte, nn_pred); nn_m.update(model="NN", X=x, train_s=round(nn_dur,1), cfg=str(nn_cfg))
    nn.save(f"{RESULTS_DIR}/models/nn_X{x}.keras")

    t = time.time(); rf, rf_cfg = train_rf(Xtr, tf_(ytr)); rf_dur = time.time()-t
    rf_pred = inv_(rf.predict(Xte))
    rf_m = evaluate(yte, rf_pred); rf_m.update(model="RF", X=x, train_s=round(rf_dur,1), cfg=str(rf_cfg))
    joblib.dump(rf, f"{RESULTS_DIR}/models/rf_X{x}.pkl")

    results += [nn_m, rf_m]
    print(f"X={x:>2} | NN R2={nn_m['R2']:.3f} ({nn_dur:.0f}s) | RF R2={rf_m['R2']:.3f} ({rf_dur:.0f}s)")

import pandas as pd
res = pd.DataFrame(results)
res.to_csv(f"{RESULTS_DIR}/metrics.csv", index=False)
res

Metrics saved to `results/metrics.csv` and models to `results/models/`. Notebook **04** compares them.